In [11]:
import os
import requests
import json
import re
from typing import List, Dict
from bs4 import BeautifulSoup
from openai import OpenAI
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from dotenv import load_dotenv

In [12]:

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [17]:
MODEL = "gpt-5-nano"
MODEL2 = "llama3.2"
openai = OpenAI(base_url = "http://localhost:11434/v1", api_key = "ollama")
#openai = OpenAI()


response = openai.chat.completions.create(model=MODEL2,messages = [{"role":"user","content":"Tell me a fun fact "}])
print(response.choices[0].message.content)


Here's one:

Did you know that honey never spoils? Archaeologists have found pots of honey in ancient Egyptian tombs that are over 3,000 years old and still perfectly edible! Honey's unique composition, with its low moisture content and high acidity, makes it an ideal food for insects and a natural preservative when stored in sealed containers.


In [3]:
def make_prompt(listings: List[Dict], user_preferences: Dict):
    prompt = (
        "You are an assistant and help a user in accommodation choosing.\n"
        "Below is a list of hotel offers and user preferences.\n"
        "HOTELS OFERS:\n"
        f"{json.dumps(listings, ensure_ascii = False, indent = 1)}\n\n"
        "USER PREFERENCES:\n"
        f"{json.dumps(user_preferences, ensure_ascii = False, indent = 1)}\n\n"
        "For every ofer:\n"
        "1) Assess suitability in 0-10 rate (where 10 = ideal suitability)\n"
        "2) Give 2-3 short reasons for your assessment\n"
        "3) Please indicate if the price is within budget. List exact price as well as preference\n"
        "4) Must include link to the property in the output\n"
        "Finally, list the TOP 3 best offers with justification.\n"
    )
    return prompt

In [4]:
def analyze_listings(listings: List[Dict], preferences: Dict):
    if not listings:
        print("No offers to analyze.")
        return None

    prompt = make_prompt(listings, preferences)

    try:
        response = openai.chat.completions.create(
            model = MODEL,
            messages = [
                {
                    "role": "system",
                    "content": "You are an expert in choosing the best accommodation.\n" 
                               "You analyze offers and advise users."
                },
                {"role": "user", "content": prompt}
            ]
        )

        result = response.choices[0].message.content
        return result

    except Exception as e:
        print(f"Communication error with LLM: {e}")
        return None

In [14]:

def main():
   # url = ("https://www.hotels.com/Hotel-Search?destination=Warsaw%20-%20Eastern%20Poland%2C%20Poland&d1=2025-10-18&startDate=2025-10-18&d2=2025-10-20&endDate=2025-10-20&adults=1&rooms=1&regionId=6057142&sort=RECOMMENDED&theme=&userIntent=&semdtl=&categorySearch=&useRewards=false&children=&latLong&pwaDialog=&daysInFuture&stayLength")

    url = ("https://www.booking.com/searchresults.en-gb.html?ss=New+South+Wales%2C+Australia&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaA-IAQGYATO4AQfIAQzYAQPoAQH4AQGIAgGoAgG4AubT7c0GwAIB0gIkNzk0YWM4MDEtZDkxZi00ZDRlLTliMDUtOGVhZmFhODQzNGY22AIB4AIB&aid=304142&lang=en-gb&sb=1&src_elem=sb&src=index&dest_id=612&dest_type=region&ac_position=0&ac_click_type=b&ac_langcode=xu&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=d8bd16b31856019b&ac_meta=GhBkOGJkMTZiMzE4NTYwMTliIAAoATICeHU6CW5ldyBzb3V0aA%3D%3D&checkin=2026-03-19&checkout=2026-03-20&group_adults=2&no_rooms=1&group_children=2&age=5&age=10")
    url2 = "https://www.airbnb.com.au/s/New-South-Wales/homes?refinement_paths%5B%5D=%2Fhomes&date_picker_type=calendar&checkin=2026-04-04&checkout=2026-04-06&adults=2&children=2&search_type=autocomplete_click&flexible_trip_lengths%5B%5D=one_week&monthly_start_date=2026-04-01&monthly_length=3&monthly_end_date=2026-07-01&price_filter_input_type=2&price_filter_num_nights=2&channel=EXPLORE&place_id=ChIJDUte93TLDWsRLZ_EIhGvgBc&location_bb=weFBlEMfGv7CFgVoQwz%2F0Q%3D%3D&acp_id=b57f9262-521c-471e-994b-a1c08d56d41e"
       
    
    preferences = {
        "max_price": 500,
        "must_have": [">= 2 beds"],
        "number_of_rooms": 2,
        "localization": "New South Wales, Australia",
        "Price":"Very Important, Lesser the better",
        "Distance": "Lesser the better",
        "Duration" : "Lesser the better",
        "Rating": "Higher the better"
    }

    print("🔍 Offers downloading from Booking.com..")
    parser = ListingParser(url2,"GlenWood, NSW")

    print(f"✅ Found {len(parser.listings)} offers\n")
    print("="*60)

    print("FOUND OFFERS:\n")
    for i, listing in enumerate(parser.listings, 1):
       # print(f"\n{i}. {listing['Name']}")
       # print(f"Amount: {listing['Price']} ")
       # print(f"Distance: {listing['Distance']} ")
       # print(f"Duration: {listing['Duration']} ")
       # print(f"Rooms: {listing['Rooms']} ")
       # print(f"Beds: {listing['Beds']} ")
       # print(f"Rating: {listing['Rating']} ")
       # print("-"*60)
        #print(f"Features: {', '.join(listing['features']) if listing['features'] else 'Informations lack.'}")

        analysis = analyze_listings(parser.listings, preferences)

        if analysis:
           print(analysis)
        else:
          print("❌ Analysis failed")

if __name__ == "__main__":
    main()
        

🔍 Offers downloading from Booking.com..
Element found successfully!<selenium.webdriver.remote.webelement.WebElement (session="4f69dd3c9326dae8af0b640dee0ff5bd", element="f.2281277C8E3D3D0BEAF608A299EB8DBD.d.2B97541C417072598047CBF3B85B8802.e.183")>
18
Cottage in Nords Wharf
Farm stay in Saumarez Ponds
Farm stay in Bellmount Forest
Cottage in Mudgee
An error occurred: Message: no such element: Unable to locate element: {"method":"css selector","selector":"span.u174bpcy.atm_7l_mhi6a6.atm_cs_80qqwn.atm_rd_14k51in.atm_rq_glywfm.atm_cs_l3jtxx__1v156lz.dir.dir-ltr"}
  (Session info: chrome=146.0.7680.164); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
0   chromedriver                        0x0000000106d7ff38 chromedriver + 6733624
1   chromedriver                        0x0000000106d776ea chromedriver + 6698730
2   chromedriver                        0x0000000106782995 chromedriver + 

KeyboardInterrupt: 

In [ ]:
%pip install geopy

In [ ]:
# get driving distance and driving time

from geopy.geocoders import Nominatim
from geopy.distance import geodesic
import time

def get_driving_time(lat1, lon1, lat2, lon2):
    # Base URL must end with a forward slash before coordinates
    # Format: lon,lat;lon,lat
    url = f"https://router.project-osrm.org/route/v1/driving/{lon1},{lat1};{lon2},{lat2}"
    params = {"overview": "false"}
    
    try:
        response = requests.get(url, params=params)
        data = response.json()
        
        if data.get('code') == 'Ok':
            # Duration is in seconds; converting to minutes
            return data['routes'][0]['duration'] / 60
        else:
            print(f"API Error: {data.get('code')}")
            return None
    except requests.exceptions.RequestException as e:
        print(f"Network Error: {e}")
        return None

# Example usage with coordinates from Bondi Beach to Parramatta
# (Using coordinates found from your previous geocoding step)
time_min = get_driving_time(-33.8915, 151.2767, -33.8150, 151.0011)
print(f"Estimated driving time: {time_min:.1f} minutes")


def find_suburb_distance(suburb1_name, suburb2_name):
    """
    Finds the distance between two suburbs using geopy.

    Args:
        suburb1_name (str): The name of the first suburb.
        suburb2_name (str): The name of the second suburb.

    Returns:
        float: The distance between the suburbs in kilometers, or None if a 
               location could not be found.
    """
    # Initialize the geolocator (Nominatim is a free service via OpenStreetMap)
    # A unique user_agent is recommended as per best practices
    geolocator = Nominatim(user_agent="suburb_distance_calculator")
    
    try:
        geoloc = {}
        # Geocode the first suburb
        print(f"Geocoding '{suburb1_name}'...")
        location1 = geolocator.geocode(suburb1_name)
        time.sleep(2) # Add delay to avoid hitting geocoding service rate limits
        
        # Geocode the second suburb
        print(f"Geocoding '{suburb2_name}'...")
        location2 = geolocator.geocode(suburb2_name)
        time.sleep(2)

        if location1 and location2:
            # Extract latitude and longitude
            coords1 = (location1.latitude, location1.longitude)
            coords2 = (location2.latitude, location2.longitude)
            
            # Calculate geodesic distance (more accurate than great-circle for Earth)
            distance = geodesic(coords1, coords2).kilometers

            geoloc["distance"] = distance
            geoloc["duration"] = get_driving_time(location1.latitude, location1.longitude, location2.latitude, location2.longitude)
            return geoloc
        else:
            print("One or both suburbs could not be geocoded.")
            return None
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

# --- Example Usage ---
suburb_a = "GlenWood, NSW"
suburb_b = "Belconnen"

loc = find_suburb_distance(suburb_a, suburb_b)


if loc is not None:
    print(f"\nThe distance between {suburb_a} and {suburb_b} is approx. {loc['distance']:.2f} km. It takes about {int(loc['duration']/ 60) } hrs and {int(loc['duration']% 60)} mins")



Estimated driving time: 39.1 minutes
Geocoding 'GlenWood, NSW'...
Geocoding 'Belconnen'...

The distance between GlenWood, NSW and Belconnen is approx. 239.71 km. It takes about 3 hrs and 27 mins


In [9]:
from seleniumwire import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import json

class ListingParser:
    def __init__(self, url, startLoc):
        
        self.listings = []
        self.url = url
        self.driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
        self.startLoc = startLoc
        self.parse()

    def parse(self):
        self.driver.get(self.url)

# 1. Find all divs

# Assuming 'driver' is your initialized WebDriver instance
        driver = self.driver

        try:
            element = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, ".g1qv1ctd.atm_u80d3j_pqaxnk.atm_c8_o7aogt.atm_g3_8jkm7i.atm_gw_iv6dct.c95t3t6.atm_9s_11p5wf0.atm_cx_1mo6erc.atm_dz_7esijk.atm_e0_1lo05zz.dir.dir-ltr"))
            )
            print(f"Element found successfully!{element}")
        except:
            print("Element not found after waiting.")


        try:
            # Use XPath to find the first direct child (represented by ./*[1])
            prop_cards = driver.find_elements(By.CSS_SELECTOR, "div[data-testid='card-container']")
            # Get and print the text
            print(len(prop_cards))

        except Exception as e:
            print(f"Could not find properties: {e}")

        # Define the CSS selector for the element
        css_selector = "span.a8jt5op.atm_3f_idpfg4.atm_7h_hxbz6r.atm_7i_ysn8ba.atm_e2_t94yts.atm_ks_zryt35.atm_l8_idpfg4.atm_mk_stnw88.atm_vv_1q9ccgz.atm_vy_t94yts.dir.dir-ltr"
        props = []

        for prop in prop_cards[:10]:

            try:
                
                deets = {}
                # Find the element using its CSS selector
                link = prop.find_element(By.CSS_SELECTOR, "a[target^='listing']").get_attribute('href')
                #print('here1')
            
                next_div = prop.find_element(By.XPATH, ".//div[contains(@class, 'lxq01kf')]")
                #print('here2')

                deets_div = next_div.find_element(By.XPATH, ".//div[contains(@class, 'g1qv1ctd')]")
                #print(f'here 3')

                
                name = deets_div.find_element(By.XPATH, ".//div[contains(@data-testid, 'listing-card-title')]").text
                print(name)

                room_deets = deets_div.find_elements(By.CSS_SELECTOR, css_selector)

                price = deets_div.find_element(By.CSS_SELECTOR,"span.u174bpcy.atm_7l_mhi6a6.atm_cs_80qqwn.atm_rd_14k51in.atm_rq_glywfm.atm_cs_l3jtxx__1v156lz.dir.dir-ltr").text
                #print(price)

                rooms = room_deets[0].text
                beds = room_deets[1].text
                rating = room_deets[5].text
            # print(f' {rooms} : {beds} :{price}' )

                # Extract the text content of the element
                
                deets["Name"] = name;
                deets["Link"] = link;
                deets["Rooms"] = rooms
                deets["Beds"] = beds;
                deets["Price"] = price;
                deets["Rating"] = rating;

                props.append(deets)
            except Exception as e:
                print(f"An error occurred: {e}")


        # Close the browser
        print(json.dumps(props, indent=4))
        driver.quit()

        for index,prop in enumerate(props):
            dest = prop["Name"].partition("in ")[2]
            prop["Location"] = f'{dest}, NSW'            
            print(dest)
            sub_dist = find_suburb_distance("GlenWood, NSW", prop["Location"])
            if sub_dist is not None:
             prop["Distance"] = f'{sub_dist["distance"]:.2f} kms'
             prop["Duration"] = f"{int(sub_dist['duration']/ 60) } hrs and {int(sub_dist['duration']% 60)} mins"
             props[index] = prop
        self.listings = props
        
#url2 = "https://www.airbnb.com.au/s/New-South-Wales/homes?refinement_paths%5B%5D=%2Fhomes&date_picker_type=calendar&checkin=2026-04-04&checkout=2026-04-06&adults=2&children=2&search_type=autocomplete_click&flexible_trip_lengths%5B%5D=one_week&monthly_start_date=2026-04-01&monthly_length=3&monthly_end_date=2026-07-01&price_filter_input_type=2&price_filter_num_nights=2&channel=EXPLORE&place_id=ChIJDUte93TLDWsRLZ_EIhGvgBc&location_bb=weFBlEMfGv7CFgVoQwz%2F0Q%3D%3D&acp_id=b57f9262-521c-471e-994b-a1c08d56d41e"

#parser = ListingParser(url2,"GlenWood, NSW")



In [ ]:
props = [{"Name":"Farm stay in Menah", "Price":"500 AUD"}]

for index,prop in enumerate(props):
            dest = prop["Name"].partition("in ")[2]
            prop["Location"] = f'{dest}, NSW'            
            print(dest)
            sub_dist = find_suburb_distance("GlenWood, NSW", prop["Location"])
            prop["Distance"] = sub_dist['distance']
print(props)